In [54]:
2+2

4

In [55]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil


In [56]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [57]:
capture_path = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/'
index_df = pd.read_csv(capture_path + 'index.csv')
print(index_df.shape)

nrows = index_df.shape[0]

# index_df expand by creating duplicate rows for each row by adding a new column 'cam': ['cam0','cam1','cam3']
index_df = index_df.loc[index_df.index.repeat(3)].reset_index(drop=True)
index_df['cam'] = ['cam0','cam1','cam2']*nrows

index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(155, 4)
(465, 5)


,video_id,template,placeholders,session,cam
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0
1,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1


In [58]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(465, 5)


,name,label,top1_pred,correct_top1,correct_top5
185,260930-122848_cam2.avi,112,112,True,True
150,260930-122254_cam0.avi,107,49,False,True


In [59]:
# I want to break video_id into video_id and cam
result[['video_id', 'cam']] = result.name.str.split('.').str[0].str.split('_', expand=True)
print(result.shape)

del result['name']

result.head(2)


(465, 7)


,label,top1_pred,correct_top1,correct_top5,video_id,cam
0,49,49,True,True,260922-134214,cam0
1,49,49,True,True,260922-134214,cam1


In [60]:
# Bring in columns with a left join
result = pd.merge(result, index_df, how = 'left', on = ['video_id','cam']) # LEFT JOIN
print(result.shape)
result.sample(3)

(465, 9)


,label,top1_pred,correct_top1,correct_top5,video_id,cam,template,placeholders,session
437,2,143,False,False,261008-114350,cam2,Bending [something] so that it deforms,nothing,mime_bending
75,107,107,True,True,260922-144349,cam0,Putting [something] next to [something],"phone, water bottle",books_2
374,69,65,False,False,261008-111617,cam2,Pretending to poke [something],nothing,mime_pretend_poke


In [61]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (465, 174)
d

(465, 174)


NpzFile './predictions.npz' with keys: logits, probabilities

# main result

In [62]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 465
Top-1 accuracy: 0.3871
Top-5 accuracy: 0.7355


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [63]:
result.groupby(['cam']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
cam,,,
cam0,155,0.35,0.69
cam1,155,0.50,0.88
cam2,155,0.31,0.63


In [64]:
result[result.cam=='cam1'].groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],8,0.12,0.50
Bending [something] so that it deforms,6,0.17,0.83
Moving [something] and [something] closer to each other,6,0.33,0.50
Moving [something] up,17,0.12,0.94
Piling [something] up,8,0.62,1.00
Plugging [something] into [something],8,0.88,1.00
Pretending to pick [something] up,6,0.83,1.00
Pretending to poke [something],9,0.33,0.78
Pretending to sprinkle air onto [something],6,0.67,1.00


In [65]:
temp = result[result.cam=='cam1'].copy()

temp.groupby( np.where(temp['session'].str.startswith('mime'), 'mime', 'control') ).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
control,137,0.53,0.91
mime,18,0.28,0.67


# Granular analysis

##### id2label

In [66]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [67]:
# Apply mapping to create new column
result['template_custom'] = result['label'].astype(str).map(id2label_dict)
result.sample(3)

,label,top1_pred,correct_top1,correct_top5,video_id,cam,template,placeholders,session,template_custom
439,2,14,False,True,261008-114415,cam1,Bending [something] so that it deforms,cardboard,bending_1,Bending [something] so that it deforms
424,68,68,True,True,261008-113443,cam1,Pretending to pick [something] up,cup,pretend_pickUp_1,Pretending to pick [something] up
232,101,101,True,True,260930-125236,cam1,Pushing [something] with [something],"photo of lady, books",lady_2,Pushing [something] with [something]


In [68]:
pd.testing.assert_series_equal(result.template, result.template_custom, check_names=False, check_exact=False)

In [69]:
np.mean(result.template == result.template_custom)*100

np.float64(100.0)